# Chapter 5 — MIMII baseline (MFCC + SVM / Random Forest)

- Input: **`fan1.zip`** (add manually in Kaggle; do not re-download or full-unzip).
- Extracts **only** `{optional_zip_root}{machine_type}/{machine_id}/` into `/kaggle/working/fan1/` (configurable).
- Baseline metrics: accuracy, precision, recall, F1, confusion matrix.
- Change **`MACHINE_TYPE`** and **`MACHINE_ID`** to reuse for pump, valve, fan, slider.

In [ ]:
# --- Configuration (edit for other machine types / IDs) ---
MACHINE_TYPE = "fan"       # e.g. "fan", "pump", "valve", "slider"
MACHINE_ID = "id_00"       # e.g. "id_00", "id_02", "id_04", "id_06"

ZIP_NAME = "fan1.zip"      # filename as uploaded to Kaggle input
EXTRACT_ROOT_NAME = "fan1"  # working folder: fan1/fan/id_00/...

# If zip entries look like "some_root/fan/id_00/...", set e.g. "some_root/"
ZIP_INNER_PREFIX = ""      # e.g. "" or "mimii_dataset/"

SAMPLE_RATE = 16_000
SEGMENT_SECONDS = 4.0      # clip length for labeling (one label per segment)
N_MFCC = 20
RANDOM_STATE = 42

TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # test = remainder

In [ ]:
import shutil
import warnings
import zipfile
from pathlib import Path

import librosa
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=UserWarning)

WORKING = Path("/kaggle/working")
KAGGLE_INPUT = Path("/kaggle/input")


def find_zip_filename(root: Path, filename: str) -> Path:
    """Locate ZIP: search `root` first (e.g. WORKING), then the other Kaggle path."""
    secondary = KAGGLE_INPUT if root == WORKING else WORKING
    for search_root in (root, secondary):
        matches = sorted(search_root.rglob(filename))
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"Could not find {filename!r} under {WORKING} or {KAGGLE_INPUT}. "
        "Ensure the file name matches ZIP_NAME."
    )


def normalize_zip_name(name: str) -> str:
    return name.replace("\\", "/").lstrip("./")


def selective_extract(zip_path: Path, prefix: str, dest_dir: Path) -> None:
    """
    Extract only members whose normalized path starts with `prefix`.
    Does not unzip the whole archive.
    """
    prefix = prefix.strip("/") + "/"
    dest_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        for m in zf.namelist():
            norm = normalize_zip_name(m)
            if norm.endswith("/"):
                continue
            if not norm.startswith(prefix):
                continue
            zf.extract(m, path=dest_dir)
    print(f"Extracted members with prefix {prefix!r} from {zip_path} -> {dest_dir}")

In [ ]:
zip_path = find_zip_filename(WORKING, ZIP_NAME)
print("Using zip:", zip_path)

extract_root = WORKING / EXTRACT_ROOT_NAME
if extract_root.exists():
    shutil.rmtree(extract_root)

_parts = [p for p in ZIP_INNER_PREFIX.replace("\\", "/").split("/") if p]
inner_prefix = "/".join(_parts + [MACHINE_TYPE, MACHINE_ID]) + "/"
selective_extract(zip_path, inner_prefix, extract_root)

audio_root = extract_root.joinpath(*_parts, MACHINE_TYPE, MACHINE_ID)
normal_dir = audio_root / "normal"
abnormal_dir = audio_root / "abnormal"

for d in (normal_dir, abnormal_dir):
    if not d.is_dir():
        raise FileNotFoundError(f"Expected directory missing: {d}")

print("Normal wavs:", len(list(normal_dir.glob("*.wav"))))
print("Abnormal wavs:", len(list(abnormal_dir.glob("*.wav"))))

In [ ]:
def wav_paths_and_labels(normal_dir: Path, abnormal_dir: Path):
    paths, labels = [], []
    for p in sorted(normal_dir.glob("*.wav")):
        paths.append(p)
        labels.append(0)  # normal
    for p in sorted(abnormal_dir.glob("*.wav")):
        paths.append(p)
        labels.append(1)  # abnormal
    return paths, np.array(labels, dtype=np.int64)


def file_to_mfcc_vectors(
    wav_path: Path,
    sr: int,
    segment_seconds: float,
    n_mfcc: int,
) -> list[np.ndarray]:
    """
    Split one wav into fixed-length segments; return one feature vector per segment
    (mean and std over time for each MFCC coeff -> length 2 * n_mfcc).
    """
    y_audio, _ = librosa.load(wav_path, sr=sr, mono=True)
    seg_len = int(segment_seconds * sr)
    if seg_len <= 0:
        raise ValueError("segment_seconds too small")

    feats = []
    for start in range(0, len(y_audio), seg_len):
        chunk = y_audio[start : start + seg_len]
        if chunk.size < seg_len:
            if chunk.size < seg_len // 2:
                continue
            chunk = np.pad(chunk, (0, seg_len - chunk.size))

        mfcc = librosa.feature.mfcc(y=chunk, sr=sr, n_mfcc=n_mfcc)
        mu = mfcc.mean(axis=1)
        sigma = mfcc.std(axis=1)
        feats.append(np.concatenate([mu, sigma]))
    return feats


paths, y_files = wav_paths_and_labels(normal_dir, abnormal_dir)
X_list, y_list = [], []

for pth, lab in zip(paths, y_files):
    for vec in file_to_mfcc_vectors(pth, SAMPLE_RATE, SEGMENT_SECONDS, N_MFCC):
        X_list.append(vec)
        y_list.append(lab)

X = np.asarray(X_list, dtype=np.float32)
y = np.asarray(y_list, dtype=np.int64)

print("Segments:", X.shape[0], "features:", X.shape[1], "positive rate:", round(y.mean(), 3))

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=(1.0 - TRAIN_FRAC),
    random_state=RANDOM_STATE,
    stratify=y,
)
val_ratio = VAL_FRAC / (VAL_FRAC + (1.0 - TRAIN_FRAC - VAL_FRAC))
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=(1.0 - val_ratio),
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print("Train/val/test:", X_train_s.shape[0], X_val_s.shape[0], X_test_s.shape[0])

In [ ]:
def report_model(name: str, clf, X_tr, y_tr, X_te, y_te):
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    print(f"\n=== {name} ===")
    print("Accuracy :", f"{accuracy_score(y_te, y_pred):.4f}")
    print("Precision:", f"{precision_score(y_te, y_pred, pos_label=1, zero_division=0):.4f}")
    print("Recall   :", f"{recall_score(y_te, y_pred, pos_label=1, zero_division=0):.4f}")
    print("F1       :", f"{f1_score(y_te, y_pred, pos_label=1, zero_division=0):.4f}")
    print("Confusion matrix [ [TN FP] [FN TP] ]:")
    print(confusion_matrix(y_te, y_pred))
    print(classification_report(y_te, y_pred, target_names=["normal", "abnormal"]))


svm = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=RANDOM_STATE)
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

report_model("SVM (RBF)", svm, X_train_s, y_train, X_test_s, y_test)
report_model("Random Forest", rf, X_train_s, y_train, X_test_s, y_test)

## Notes

- **Zip layout**: members must match `ZIP_INNER_PREFIX` + `{MACHINE_TYPE}/{MACHINE_ID}/normal|abnormal/*.wav`.
- **Validation**: `X_val` is reserved for future hyperparameter tuning; baselines are fit on train only and scored on test.
- **Slider**: MIMII folder name is often `slider` not `slide`; use `MACHINE_TYPE = "slider"` if that matches your zip.